In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()

In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

digit_recognizer_path = kagglehub.competition_download('digit-recognizer')

print('Data source import complete.')


Data source import complete.


In [3]:
# ---- import Libraries ----
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import Flatten

from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dropout


from sklearn.linear_model import Perceptron

from tensorflow.keras.utils import to_categorical

In [4]:
# ---- Importing train and test ----
import os

print(os.listdir(digit_recognizer_path))  # sanity check - see what files are there

df = pd.read_csv(os.path.join(digit_recognizer_path, 'train.csv'))
df_test = pd.read_csv(os.path.join(digit_recognizer_path, 'test.csv'))

print(df.shape, df_test.shape)
df.head()

['sample_submission.csv', 'train.csv', 'test.csv']
(42000, 785) (28000, 784)


,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
# ---- test data ----
df_test.head()

,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
df.isna().sum()

label       0
pixel0      0
pixel1      0
pixel2      0
pixel3      0
           ..
pixel779    0
pixel780    0
pixel781    0
pixel782    0
pixel783    0
Length: 785, dtype: int64

In [7]:
# ---- shape ans sizes ----
print(df.columns)
print(df.shape)
print(df_test.shape)
print(df.info())

Index(['label', 'pixel0', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5',
       'pixel6', 'pixel7', 'pixel8',
       ...
       'pixel774', 'pixel775', 'pixel776', 'pixel777', 'pixel778', 'pixel779',
       'pixel780', 'pixel781', 'pixel782', 'pixel783'],
      dtype='object', length=785)
(42000, 785)
(28000, 784)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42000 entries, 0 to 41999
Columns: 785 entries, label to pixel783
dtypes: int64(785)
memory usage: 251.5 MB
None


In [8]:
# ---- Preprocessing ----

X = df.drop('label', axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)   # Split into train and test

X_train = X_train.astype('float32')/ 255.0      # Scaling
X_test = X_test.astype('float32')/ 255.0

y_train = to_categorical(y_train, num_classes=10)    # OneHot encoding
y_test = to_categorical(y_test, num_classes=10)

X_train_cnn =  X_train.values.reshape(-1, 28, 28, 1)  # The last num is num of layers 1 for black and white.
X_test_cnn =  X_test.values.reshape(-1, 28, 28, 1)

In [9]:
# ---- CNN Modeling ----
model = Sequential([
    Conv2D(filters = 32, kernel_size = (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),
    Conv2D(filters = 64, kernel_size = (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

model.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

history = model.fit(X_train_cnn, y_train, epochs = 5, batch_size = 64,
                    validation_data = (X_test_cnn, y_test),verbose = 1)

2026-07-25 10:42:36.654941: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/5
525/525 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - accuracy: 0.8540 - loss: 0.4595 - val_accuracy: 0.9736 - val_loss: 0.0884
Epoch 2/5
525/525 ━━━━━━━━━━━━━━━━━━━━ 15s 28ms/step - accuracy: 0.9496 - loss: 0.1644 - val_accuracy: 0.9799 - val_loss: 0.0638
Epoch 3/5
525/525 ━━━━━━━━━━━━━━━━━━━━ 14s 27ms/step - accuracy: 0.9641 - loss: 0.1185 - val_accuracy: 0.9848 - val_loss: 0.0502
Epoch 4/5
525/525 ━━━━━━━━━━━━━━━━━━━━ 14s 27ms/step - accuracy: 0.9697 - loss: 0.1014 - val_accuracy: 0.9861 - val_loss: 0.0479
Epoch 5/5
525/525 ━━━━━━━━━━━━━━━━━━━━ 14s 27ms/step - accuracy: 0.9751 - loss: 0.0840 - val_accuracy: 0.9860 - val_loss: 0.0410


In [10]:
acc = model.evaluate(X_test_cnn, y_test, verbose=0)
print('Test Accuracy: %.3f' % acc[1])

Test Accuracy: 0.986


In [11]:
# ---- preprocessing test data ----
df_test = df_test.astype('float32')/ 255.0
df_test = df_test.values.reshape(-1, 28, 28, 1)

# ---- Generate predictions ----
predictions = model.predict(df_test)
predicted_labels = np.argmax(predictions, axis=1)   # convert probabilities -> class label (0-9)

# ---- Build submission dataframe ----
submission = pd.DataFrame({
    'ImageId': range(1, len(predicted_labels) + 1),
    'Label': predicted_labels
})

# ---- Save to CSV ----
submission.to_csv('submission.csv', index=False)

print(submission.head())

875/875 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step
   ImageId  Label
0        1      2
1        2      0
2        3      9
3        4      9
4        5      3
